# ⚗️ DSA-Mamba (Dense Spectral Attention Mamba) — Eye Image HB Estimation
**Custom model** built for this exact use case: retinal/conjunctival image → HB estimation.

**Key innovation:** Dense Spectral Attention layer weights RGB channels by  
their frequency-domain energy — capturing hemoglobin-specific spectral signatures.

**Tasks:** ① Binary classification ② HB regression


In [ ]:
import subprocess, sys
pkgs = ["einops","timm","pandas","openpyxl","scikit-learn","matplotlib","tqdm"]
for p in pkgs:
    subprocess.run([sys.executable,"-m","pip","install",p,"-q"], check=False)
print("✅ Done")


In [ ]:
# Copy dsa_mamba.py from the cloned repo (or paste it inline)
import subprocess
# If running from cloned repo structure:
# sys.path.insert(0, "/kaggle/working/DSA_Mamba_Custom")
# from dsa_mamba import DSAMamba, DSAMambaLoss

# ── Or paste inline (self-contained) ────────────────────────────────────────
import math, os, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from einops import rearrange, repeat
from timm.models.layers import DropPath, to_2tuple, trunc_normal_
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_absolute_error
import matplotlib.pyplot as plt
from tqdm import tqdm
warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ── Paste dsa_mamba.py content ──────────────────────────────────────────────
# (Model code from 07_DSA_Mamba_Custom/dsa_mamba.py is included below)
# ──────────────────────────────────────────────────────────────────────────────
exec(open("/kaggle/working/DSA_Mamba_Custom/dsa_mamba.py").read()
     if os.path.exists("/kaggle/working/DSA_Mamba_Custom/dsa_mamba.py")
     else open("dsa_mamba.py").read())
print("✅ DSA-Mamba model loaded")


In [ ]:
# Config
CFG = dict(
    image_dir  = "/kaggle/input/datasets/junaidgpu/imagehb/dataset/dataset/left_eye",
    csv_path   = "/kaggle/input/datasets/junaidgpu/imagehb/dataset/dataset/merge_excel_1.xlsx",
    image_col  = "Patient ID",
    hb_col     = "HB",
    hb_thresh  = 12.0,
    img_size   = 224, patch_size=16,
    embed_dim  = 192, depth=8,
    epochs     = 40, batch_size=8, lr=1e-4,
    cls_weight = 1.0, reg_weight=0.5,
    val_split  = 0.2, seed=42,
)

class EyeHBDataset(Dataset):
    def __init__(self, df, image_dir, image_col, hb_col, hb_thresh, transform=None):
        self.df=df.reset_index(drop=True); self.image_dir=image_dir
        self.image_col=image_col; self.hb_col=hb_col
        self.hb_thresh=hb_thresh; self.transform=transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]; pid=str(row[self.image_col]); hb=float(row[self.hb_col])
        label=0 if hb < self.hb_thresh else 1
        for ext in [".jpg",".jpeg",".png",""]:
            p=os.path.join(self.image_dir, pid+ext)
            if os.path.exists(p): break
        img=Image.open(p).convert("RGB")
        if self.transform: img=self.transform(img)
        return img, torch.tensor(label,dtype=torch.long), torch.tensor([[hb]],dtype=torch.float32)

T_train = transforms.Compose([transforms.Resize((256,256)),transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),transforms.ColorJitter(0.3,0.3,0.2),
    transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
T_val   = transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),
    transforms.Normalize([.485,.456,.406],[.229,.224,.225])])

df=pd.read_excel(CFG["csv_path"])
train_df,val_df=train_test_split(df,test_size=CFG["val_split"],random_state=CFG["seed"],
    stratify=(df[CFG["hb_col"]]<CFG["hb_thresh"]).astype(int))
train_loader=DataLoader(EyeHBDataset(train_df,CFG["image_dir"],CFG["image_col"],
    CFG["hb_col"],CFG["hb_thresh"],T_train),batch_size=CFG["batch_size"],shuffle=True,num_workers=2)
val_loader=DataLoader(EyeHBDataset(val_df,CFG["image_dir"],CFG["image_col"],
    CFG["hb_col"],CFG["hb_thresh"],T_val),batch_size=CFG["batch_size"],shuffle=False,num_workers=2)
print(f"Train: {len(train_df)} | Val: {len(val_df)}")


In [ ]:
# Model + train
model = DSAMamba(img_size=CFG["img_size"], patch_size=CFG["patch_size"],
                  embed_dim=CFG["embed_dim"], depth=CFG["depth"]).to(DEVICE)
print(f"DSA-Mamba Params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M")

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["epochs"])
criterion = DSAMambaLoss(CFG["cls_weight"], CFG["reg_weight"])

best=float("inf")
for epoch in range(1, CFG["epochs"]+1):
    model.train(); tl=0
    for imgs,labels,hb in tqdm(train_loader,desc=f"DSA-Mamba Ep{epoch}",leave=False):
        imgs,labels,hb=imgs.to(DEVICE),labels.to(DEVICE),hb.to(DEVICE)
        optimizer.zero_grad()
        lo,hr=model(imgs); loss,_,_=criterion(lo,hr,labels,hb)
        loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.); optimizer.step()
        tl+=loss.item()
    scheduler.step(); tl/=len(train_loader)
    model.eval(); vl=correct=total=0; hbp=[]; hbt=[]
    with torch.no_grad():
        for imgs,labels,hb in val_loader:
            imgs,labels,hb=imgs.to(DEVICE),labels.to(DEVICE),hb.to(DEVICE)
            lo,hr=model(imgs); loss,_,_=criterion(lo,hr,labels,hb)
            vl+=loss.item(); correct+=(lo.argmax(1)==labels).sum().item()
            total+=labels.size(0); hbp+=hr.cpu().squeeze().tolist(); hbt+=hb.cpu().squeeze().tolist()
    vl/=len(val_loader)
    if vl<best: best=vl; torch.save(model.state_dict(),"best_dsa_mamba.pth")
    if epoch%5==0 or epoch==1:
        print(f"Ep{epoch:3d}|TL:{tl:.4f}|VL:{vl:.4f}|Acc:{correct/total:.3f}|MAE:{mean_absolute_error(hbt,hbp):.2f}")

print(f"\n✅ Best val loss: {best:.4f}")


In [ ]:
# Final report
model.load_state_dict(torch.load("best_dsa_mamba.pth", map_location=DEVICE))
model.eval()
all_preds,all_labels,all_hbp,all_hbt=[],[],[],[]
with torch.no_grad():
    for imgs,labels,hb in val_loader:
        lo,hr=model(imgs.to(DEVICE))
        all_preds+=lo.argmax(1).cpu().tolist(); all_labels+=labels.tolist()
        all_hbp+=hr.cpu().squeeze().tolist(); all_hbt+=hb.cpu().squeeze().tolist()
print("Classification Report:")
print(classification_report(all_labels,all_preds,target_names=["Anemic","Normal"]))
print(f"HB MAE : {mean_absolute_error(all_hbt,all_hbp):.4f} g/dL")
print(f"HB RMSE: {np.sqrt(np.mean((np.array(all_hbt)-np.array(all_hbp))**2)):.4f} g/dL")
